### Importing Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import math
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

### Importing Data

In [ ]:
df = pd.read_excel("IITB_UG_Student_Dataset.xlsx")

### Cleansing Datatypes

In [ ]:
df['Exercise_frequency'] = df['Exercise_frequency'].astype(str).str.replace(r'[^0-9.]', '', regex=True)
df['Exercise_frequency'] = pd.to_numeric(df['Exercise_frequency'], errors='coerce')

### Handling Missing Values

In [ ]:
numerical_col = df.select_dtypes('float64').columns
category_col = df.select_dtypes('object').columns
category_col = [col for col in category_col if col != 'Student_ID']

for col in numerical_col:
    df[col] = df[col].fillna(df[col].median())
for col in category_col:
    df[col] = df[col].fillna(df[col].mode()[0])

### Statistical Dictionaries

In [ ]:
mean_dict = df[numerical_col].mean().to_dict()
median_dict = df[numerical_col].median().to_dict()
mode_dict = {}

for col in category_col:
    mode_val = df[col].mode(dropna=True)
    
    if not mode_val.empty:
        mode_dict[col] = mode_val[0]
    else:
        mode_dict[col] = None

print(mean_dict)
print(median_dict)
print(mode_dict)

### Visualization of Plots

In [ ]:
# Frequecy distribution of given indicators
n = len(numerical_col)
cols = 4   # number of plots per row
rows = math.ceil(n / cols)

plt.figure(figsize=(cols*5, rows*4))

for i, col in enumerate(numerical_col, 1):
    plt.subplot(rows, cols, i)
    plt.hist(df[col].dropna(), bins=20)
    plt.title(col)
    plt.xlabel("")
    plt.ylabel("")

plt.tight_layout()
plt.show()


In [ ]:
# Boxplots for analyzing outliers

n = len(numerical_col)
rows = math.ceil(n / cols)

plt.figure(figsize=(cols*5, rows*4))
for i, col in enumerate(numerical_col, 1):
    plt.subplot(rows, cols, i)
    plt.boxplot(df[col].dropna())
    plt.title(col, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots

target = 'Cumulative_Grade'

feat_cols = [c for c in numerical_col if c != target]
n = len(feat_cols)
rows = math.ceil(n / cols)

plt.figure(figsize=(cols*5, rows*4))
for i, col in enumerate(feat_cols, 1):
    plt.subplot(rows, cols, i)
    plt.scatter(df[col], df[target])
    plt.title(f"{col} vs {target}", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap

corr = df[numerical_col].corr()

plt.figure(figsize=(8,6))
plt.imshow(corr)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

### Linear Regression

In [ ]:
X = df.drop(columns=[target, 'Student_ID'], errors='ignore')
X = pd.get_dummies(X, drop_first=True)
Y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=31)

lnr = LinearRegression()
lnr.fit(X_train, y_train)

y_pred_lnr = lnr.predict(X_test)
r2_lnr = r2_score(y_test, y_pred_lnr)
rmse_lnr = math.sqrt(mean_squared_error(y_test, y_pred_lnr))
mae_lnr = mean_absolute_error(y_test, y_pred_lnr)
print(r2_lnr)
print(rmse_lnr)
print(mae_lnr)

coeff = lnr.coef_
features = X.columns

coef_df_lnr = pd.DataFrame({
    'Feature': features,
    'Coefficient': coeff
})
coef_df_lnr = coef_df_lnr.sort_values(by='Coefficient', key=abs, ascending=False)
print(coef_df_lnr)


### Random Forest Regressor

In [ ]:
rfr = RandomForestRegressor()
rfr.fit(X_train, y_train)

y_pred_rfr = rfr.predict(X_test)
r2_rfr = r2_score(y_test, y_pred_rfr)
rmse_rfr = math.sqrt(mean_squared_error(y_test, y_pred_rfr))
mae_rfr = mean_absolute_error(y_test, y_pred_rfr)

print(r2_rfr)
print(rmse_rfr)
print(mae_rfr)

importance = rfr.feature_importances_

rfr_importance = pd.DataFrame({
    'Feature': features,
    'Coefficient': importance
})
rfr_importance = rfr_importance.sort_values(by='Coefficient', ascending=False)
print(rfr_importance)

### Plots and Error Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Linear Regression: Predicted vs Actual
axes[0,0].scatter(y_test, y_pred_lnr)
axes[0,0].set_title("Linear Regression: Predicted vs Actual")
axes[0,0].set_xlabel("Actual")
axes[0,0].set_ylabel("Predicted")
axes[0,0].grid(True)

# Random Forest: Predicted vs Actual
axes[0,1].scatter(y_test, y_pred_rfr)
axes[0,1].set_title("Random Forest: Predicted vs Actual")
axes[0,1].set_xlabel("Actual")
axes[0,1].set_ylabel("Predicted")
axes[0,1].grid(True)

# Linear Regression Residual Distribution
res_lr = y_test - y_pred_lnr

axes[1,0].hist(res_lr, bins=20)
axes[1,0].set_title("LR Residual Distribution")
axes[1,0].set_xlabel("Error")

# Random Forest Residual Distribution
res_rf = y_test - y_pred_rfr

axes[1,1].hist(res_rf, bins=20)
axes[1,1].set_title("RF Residual Distribution")
axes[1,1].set_xlabel("Error")

# Adjust spacing
plt.tight_layout()

plt.show()